In [ ]:
from esapp import PowerWorld
from esapp.components import *

from esa import SAW
import pandas as pd

In [ ]:
CASE_PATH = r"D:\\Project\\OneDrive - Texas A&M University\\Desktop\\Research Project\\Cases Files\\ERCOT\\ver_30U1_2025-11-14\\ERCOT-SS-25SSWG_2026_SUM1_U1_OnPeak_Pass_Final_11132025_Set3-PWver24.pwb"
#CASE_PATH = r"D:\\Project\\OneDrive - Texas A&M University\\Desktop\\Research Project\\Cases Files\\Synthetic_grid\\Texas2k_series24_cases_with_dynamics\\Texas2k_series25_case1_summerpeak\\Texas2k_series25_case1_summerpeak_weather_county.pwb"

In [ ]:
data = PowerWorld(CASE_PATH)

In [ ]:
saw = SAW(CASE_PATH)


In [ ]:
load_params = saw.get_key_field_list("Load") + ['SubNum','LoadSMW','BusNomVolt']
load_df = saw.GetParametersMultipleElement("Load", load_params)
load_df

In [ ]:
substation_params = saw.get_key_field_list('Substation') + ['CustomString', 'CustomString:1', 'Latitude', 'Longitude','BGNominalkvRange:1','BusLoadNum','GenNum']
substation = saw.GetParametersMultipleElement('Substation', substation_params)
substation

In [ ]:
# ── Load Voltage Distribution Analysis (Texas 2k Benchmark) ───────────────────
import numpy as np

# Parse max kV from BGNominalkvRange:1
def parse_max_kv(kv_str):
    try:
        kvs = [float(x) for x in str(kv_str).split() if float(x) > 0]
        return max(kvs) if kvs else None
    except:
        return None

substation['MaxKV'] = substation['BGNominalkvRange:1'].apply(parse_max_kv)

load_df['LoadSMW'] = pd.to_numeric(load_df['LoadSMW'], errors='coerce').fillna(0)
load_df['BusNomVolt'] = pd.to_numeric(load_df['BusNomVolt'], errors='coerce').fillna(0)

# Map 115 kV to 138 kV (Texas 2k uses 115; standard ERCOT is 138)
load_df['BusNomVolt'] = load_df['BusNomVolt'].replace(115, 138)
substation['MaxKV'] = substation['MaxKV'].replace(115, 138)

# Filter to transmission-level loads (>= 69 kV)
tx_loads = load_df[load_df['BusNomVolt'] >= 69].copy()

# ── Cross-tabulation: Load Bus kV vs Substation Max kV ────────────────────────
substation['SubNum'] = substation['SubNum'].astype(str)
tx_loads['SubNum'] = tx_loads['SubNum'].astype(str)
loads_with_sub = tx_loads.merge(substation[['SubNum', 'MaxKV']], on='SubNum', how='left')
loads_with_sub = loads_with_sub[loads_with_sub['MaxKV'].notna() & (loads_with_sub['MaxKV'] >= 69)]

# Row-normalized: for each sub max kV, % of loads at each bus kV
cross_pct = pd.crosstab(
    loads_with_sub['MaxKV'].astype(int).rename('Sub Max kV'),
    loads_with_sub['BusNomVolt'].astype(int).rename('Load Bus kV'),
    normalize='index'
).multiply(100).round(0)

print("── By Load Count ──")
print(f"{'Sub Max kV':<12} {'→ 69 kV':>10} {'→ 138 kV':>10} {'→ 345 kV':>10}")
print("-"*44)
for sub_kv in sorted(cross_pct.index):
    if sub_kv in [69, 138, 345]:
        row = cross_pct.loc[sub_kv]
        print(f"{sub_kv:<12} {row.get(69, 0):>9.0f}% {row.get(138, 0):>9.0f}% {row.get(345, 0):>9.0f}%")

# ── Load size statistics by bus voltage ───────────────────────────────────────
print("\n── Load Size by Voltage Level ──")
print(f"{'Bus kV':<10} {'Median':>8} {'Mean':>8} {'Max':>8}   {'Description'}")
print("-"*60)
desc_map = {69: 'Small distribution loads', 138: 'Bulk system loads', 345: 'Large industrial / data center'}
for kv in [69, 138, 345]:
    vals = tx_loads[tx_loads['BusNomVolt'] == kv]['LoadSMW']
    if len(vals) > 0:
        print(f"{kv:<10} {vals.median():>7.1f} {vals.mean():>7.1f} {vals.max():>7.0f}   {desc_map.get(kv, '')}")

# ── Substation Load Voltage Combination Distribution ──────────────────────────
# For each substation with load, find which voltage levels have loads
# e.g., "138 only", "138 + 69", "345 + 138 + 69", etc.

# Get unique load bus kV levels per substation
sub_load_kvs = loads_with_sub.groupby('SubNum')['BusNomVolt'].apply(
    lambda x: tuple(sorted(set(int(v) for v in x), reverse=True))
).reset_index()
sub_load_kvs.columns = ['SubNum', 'LoadKVCombination']

# Count substations per combination
combo_counts = sub_load_kvs['LoadKVCombination'].value_counts().sort_index()

total_subs_with_load = len(sub_load_kvs)

print("\n── Substation Load Voltage Combinations ──")
print(f"{'Combination':<25} {'Subs':>6} {'%':>7}")
print("-"*40)
for combo, count in combo_counts.items():
    label = ' + '.join(f"{kv} kV" for kv in combo)
    pct = 100 * count / total_subs_with_load
    print(f"{label:<25} {count:>6} {pct:>6.1f}%")
print("-"*40)
print(f"{'Total':<25} {total_subs_with_load:>6}")

# Also show with MW totals per combination
sub_load_mw = loads_with_sub.groupby('SubNum').agg(
    TotalMW=('LoadSMW', 'sum'),
    LoadKVs=('BusNomVolt', lambda x: tuple(sorted(set(int(v) for v in x), reverse=True)))
).reset_index()

combo_mw = sub_load_mw.groupby('LoadKVs').agg(
    Subs=('SubNum', 'count'),
    TotalMW=('TotalMW', 'sum')
).sort_index()

total_mw_all = combo_mw['TotalMW'].sum()

print("\n── With MW breakdown ──")
print(f"{'Combination':<25} {'Subs':>6} {'% Subs':>7} {'MW':>10} {'% MW':>7}")
print("-"*58)
for combo, row in combo_mw.iterrows():
    label = ' + '.join(f"{kv} kV" for kv in combo)
    pct_s = 100 * row['Subs'] / total_subs_with_load
    pct_m = 100 * row['TotalMW'] / total_mw_all
    print(f"{label:<25} {row['Subs']:>6} {pct_s:>6.1f}% {row['TotalMW']:>10,.0f} {pct_m:>6.1f}%")
print("-"*58)
print(f"{'Total':<25} {total_subs_with_load:>6}         {total_mw_all:>10,.0f}")


ESAPP FIx: Need to update the substation parameter: Missing quite a lot of stuffs.


What I am seeing is some of the 345 kV with very low load MW also have 345 kV load with 3 MW load capacity. This seems wrong. If 345 kV has low MW you can either put 69 kV load or 135 kV load if the number fits the load mw